# 04 - Robot camera, marker detection and pose accuracy

Renders the robot-mounted camera at a grid of robot poses, detects the ArUco marker, estimates the marker pose in the robot frame and compares it with the true value computed from ground truth. Ground truth is used for evaluation only. The camera mounting and field of view are placeholders until the lab values are known.

In [ ]:
!pip install -q mujoco opencv-contrib-python
!test -d robot || git clone -q https://github.com/hongjiezheng-1-hue/robot.git
!cd robot && git pull -q
!test -d robotis_mujoco_menagerie || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ROBOTIS-GIT/robotis_mujoco_menagerie.git
!cd robotis_mujoco_menagerie && git sparse-checkout set robotis_tb3

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
import sys
sys.path.insert(0, "robot")
import numpy as np
import cv2
import mujoco
import matplotlib.pyplot as plt
from sim.scene import SceneConfig, write_scene
from perception.aruco import detect_markers
from perception.marker_pose import camera_intrinsics, estimate_marker_pose
from estimation.models import h_marker_pose, wrap

print("OpenCV", cv2.__version__, "| MuJoCo", mujoco.__version__)
cfg = SceneConfig()
path = write_scene(cfg, "robotis_mujoco_menagerie/robotis_tb3", flip_marker=False)
model = mujoco.MjModel.from_xml_path(path)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=cfg.camera_height, width=cfg.camera_width)
K = camera_intrinsics(cfg.camera_fovy, cfg.camera_width, cfg.camera_height)
MARKER = (cfg.station_x, cfg.station_y, cfg.station_yaw)
print("cameras in the model:", model.ncam)

In [ ]:
def view(x, y, yaw):
    mujoco.mj_resetData(model, data)
    data.qpos[0], data.qpos[1] = x, y
    data.qpos[3], data.qpos[6] = np.cos(yaw / 2), np.sin(yaw / 2)   # free-joint quaternion (w, x, y, z)
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=cfg.camera_name)
    return renderer.render().copy()

def measure(img):
    ids, corners = detect_markers(img, cfg.marker_dictionary)
    if ids is None or cfg.marker_id not in ids.flatten():
        return None, None
    c = corners[list(ids.flatten()).index(cfg.marker_id)].reshape(4, 2)
    return estimate_marker_pose(c, K, cfg.marker_size, cfg.camera_pos), c

rows = []
for x in (0.25, 0.65, 1.05, 1.45):
    for y in (-0.3, 0.0, 0.3):
        for dyaw in (-0.2, 0.0, 0.2):
            yaw = np.arctan2(-y, cfg.station_x - x) + dyaw    # roughly facing the marker
            est, _ = measure(view(x, y, yaw))
            dist = np.hypot(cfg.station_x - x, cfg.station_y - y)
            if est is None:
                rows.append((dist, False, np.nan, np.nan))
            else:
                truth = h_marker_pose(np.array([x, y, yaw]), MARKER)
                rows.append((dist, True, 1000 * np.hypot(*(est[:2] - truth[:2])), np.degrees(abs(wrap(est[2] - truth[2])))))
rows = np.array(rows, dtype=float)

print("distance band [m] | poses | detected | position error median/95% [mm] | heading error median/95% [deg]")
for lo, hi in ((0.2, 0.7), (0.7, 1.1), (1.1, 1.5), (1.5, 1.9)):
    sel = rows[(rows[:, 0] >= lo) & (rows[:, 0] < hi)]
    if len(sel) == 0:
        continue
    det = sel[sel[:, 1] == 1]
    if len(det) == 0:
        print(f"{lo:.1f} - {hi:.1f}         | {len(sel):5d} | {0:8d} | no detections")
        continue
    print(f"{lo:.1f} - {hi:.1f}         | {len(sel):5d} | {len(det):8d} | {np.median(det[:, 2]):7.1f} / {np.percentile(det[:, 2], 95):7.1f}"
          f"          | {np.median(det[:, 3]):6.1f} / {np.percentile(det[:, 3], 95):6.1f}")

In [ ]:
img = view(1.05, 0.0, 0.0)
est, c = measure(img)
vis = img.copy()
if c is not None:
    cv2.polylines(vis, [c.astype(np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 2)
plt.figure(figsize=(6, 4.5))
plt.imshow(vis)
plt.axis("off")
plt.title("robot camera at (1.05, 0, 0)")
plt.show()
print("estimated [x, y, relative heading] in the robot frame:", est)
print("true value:", h_marker_pose(np.array([1.05, 0.0, 0.0]), MARKER))

## What to look for

- The model should contain one camera. Detection should work at the shorter distances; note where it stops.
- Position error should be small compared with the marker distance; heading error grows with distance (planar pose ambiguity, see docs/findings.md).
- The estimate and the true value in the last cell should be close.